<a href="https://colab.research.google.com/github/qweqwe24011-debug/AI_Puzzle_Restoration/blob/main/Vinalpw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Обоснование архитектуры и выбор подхода

Данная задача состоит из двух взаимосвязанных подзадач: **сборка пазла** и **реставрация изображения**. Простой baseline не дает высокого SSIM, потому что даже при идеальной сборке между фрагментами остаются видимые швы из-за индивидуальных искажений (разная яркость, контраст, шум, JPEG-артефакты).

### 1. Сборка пазла (Puzzle Solving)
- **Архитектура**: Компактная сверточная сеть (`PairCNN`) с Batch Normalization. На вход подается канва 40×40 (два соседних фрагмента). Сеть учится оценивать непрерывность границ и текстур, игнорируя локальные изменения яркости и шума.
- **Умный выбор старта**: Жадный алгоритм сборки критически зависит от выбора начального фрагмента. Мы вычисляем `corner_score` (разница между исходящими и входящими вероятностями связей фрагмента) и перебираем топ-30 кандидатов на роль верхнего левого угла. Выбирается та сборка, которая имеет максимальную внутреннюю согласованность (max mean score).

### 2. Реставрация изображения (Image Restoration)
- **Ансамбль моделей**: Вместо одной сети используется ансамбль из 3-х различных архитектур (`UNet`, `ResNet`, `WideUNet`):
  - **UNet**: Отлично восстанавливает высокочастотные детали и сглаживает швы благодаря skip-connections, передающим информацию с разных уровней абстракции.
  - **ResNet** (с residual learning): Стабильно вычитает шум и артефакты, не размывая исходные текстуры изображения.
  - **WideUNet**: Захватывает более широкий глобальный контекст, помогая выровнять общую освещенность между фрагментами.
- **Усреднение предсказаний**: Выходы всех трех моделей усредняются. Это классический прием (variance reduction), который значительно снижает дисперсию ошибок и подавляет артефакты, характерные для отдельных архитектур.

### 3. Синтетические аугментации (Domain Adaptation)
Ключ к высокому SSIM – обучение на данных, максимально приближенных к тестовым. Функция `corrupt_fragment` в точности воспроизводит условия конкурса:
1. Контраст: 0.70–1.30
2. Яркость: ±30
3. Размытие: Gaussian Blur (ядро ~3×3)
4. JPEG-сжатие: quality 35–50
5. Аддитивный гауссовский шум: σ = 40–55

Применение этих искажений *на лету* к чистым фрагментам во время обучения заставляет модели учиться истинным признакам соседства и очистки, а не просто запоминать датасет.

### 4. Строгая воспроизводимость (Reproducibility)
Для выполнения жестких требований конкурса и получения бит-в-бит одинакового результата на любой видеокарте (например, Colab T4 vs локальная RTX) реализована полная фиксация всех источников случайности:
- `random.seed`, `np.random.seed`, `torch.manual_seed`
- Включен детерминированный режим `cuDNN`: `torch.backends.cudnn.deterministic = True` и `benchmark = False`.
- Фиксация `PYTHONHASHSEED`.

In [ ]:
# ==========================================================
# 1. УСТАНОВКА ВСЕХ ЗАВИСИМОСТЕЙ
# ==========================================================
!pip install -q pillow numpy scikit-image torch torchvision matplotlib tqdm pandas gdown

In [ ]:
# ==========================================================
# 2. ИМПОРТ, ФИКСАЦИЯ SEED И АВТООПРЕДЕЛЕНИЕ ОКРУЖЕНИЯ
# ==========================================================
import os
import random
import zipfile
import io
import time
import pandas as pd
from datetime import datetime

import numpy as np
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ==========================================================
# ФИКСАЦИЯ СЛУЧАЙНОСТИ (ТРЕБОВАНИЕ КОНКУРСА)
# ==========================================================
def set_seed(seed=42):
    """
    Полная фиксация случайности для воспроизводимости.
    Включает детерминированный режим cuDNN, чтобы результат
    был одинаковым на любой видеокарте.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # КРИТИЧНО для воспроизводимости на разных GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

SEED = 42
set_seed(SEED)

# ==========================================================
# АВТООПРЕДЕЛЕНИЕ ОКРУЖЕНИЯ (ЛОКАЛЬНО / GOOGLE COLAB)
# ==========================================================
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🚀 Запуск в Google Colab. Скачиваю датасет...")
    import gdown

    # ID архива из Google Drive
    FILE_ID = "1sd-SFX38tosFI6yYQLetuM7Wp_tIbIlY"
    ZIP_PATH = "dataset.zip"

    # Скачиваем, если архива нет
    if not os.path.exists(ZIP_PATH):
        print("📥 Скачиваю архив...")
        gdown.download(id=FILE_ID, output=ZIP_PATH, quiet=False)

    # Распаковываем, если папки ещё нет
    if not os.path.exists("./dataset/train"):
        print("📦 Распаковываю архив...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(".")

    DATA_DIR = "./dataset"
    print("✅ Данные готовы в ./dataset")
else:
    # Локальный запуск на вашем ПК
    DATA_DIR = r"D:\prog\python_p\ii\2p_2"
    print(f"💻 Локальный запуск. Путь: {DATA_DIR}")

# ==========================================================
# ПУТИ К ПАПКАМ
# ==========================================================
GRID = 24
FS = 20
IMG_SIZE = GRID * FS  # 480

TRAIN_INPUT_DIR = os.path.join(DATA_DIR, "train", "inputs")
TRAIN_TARGET_DIR = os.path.join(DATA_DIR, "train", "targets")
TEST_DIR = os.path.join(DATA_DIR, "test")

SUBMISSION_DIR = os.path.join(DATA_DIR, "submission_final")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# ==========================================================
# КОНФИГУРАЦИЯ ЭКСПЕРИМЕНТА (ПАЙПЛАЙН ГИПЕРПАРАМЕТРОВ)
# ==========================================================
# Для быстрой проверки поставьте QUICK_MODE = True
# Для финальной сдачи оставьте QUICK_MODE = False
QUICK_MODE = False  # ← Поставить True для быстрой проверки

if QUICK_MODE:
    CONFIG = {
        "val_size": 100,
        "puzzle_max_images": 100,
        "puzzle_epochs": 3,
        "puzzle_batch_size": 128,
        "puzzle_num_starts": 10,

        "restore_max_images": 100,
        "restore_epochs": 3,
        "restore_batch_size": 4,

        "eval_images": 3,
        "submission_num_starts": 10,
    }
    RUN_FULL_SUBMISSION = False
    print("⚡ Быстрый режим (для отладки)")
else:
    CONFIG = {
        "val_size": 500,
        "puzzle_max_images": 1000,
        "puzzle_epochs": 8,
        "puzzle_batch_size": 128,
        "puzzle_num_starts": 30,

        "restore_max_images": 1000,
        "restore_epochs": 15,
        "restore_batch_size": 4,

        "eval_images": 5,
        "submission_num_starts": 30,
    }
    RUN_FULL_SUBMISSION = True
    print("🏆 Финальный режим (полный запуск)")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Device: {device}")
print(f"🎯 Полный сабмит: {RUN_FULL_SUBMISSION}")

In [ ]:
# ==========================================================
# 3. АНАЛИЗ ДАННЫХ (ОБОСНОВАНИЕ ВЫБОРА ПОДХОДА)
# ==========================================================
def load_image(path):
    """Загружает PNG как RGB-массив"""
    return np.array(Image.open(path).convert("RGB"))

def get_png_names(directory):
    """Список всех PNG в папке"""
    return sorted([n for n in os.listdir(directory) if n.lower().endswith(".png")])

# Проверяем данные
input_names = get_png_names(TRAIN_INPUT_DIR)
target_names = get_png_names(TRAIN_TARGET_DIR)
test_names = get_png_names(TEST_DIR)

print("=" * 50)
print("📊 АНАЛИЗ ДАННЫХ")
print("=" * 50)
print(f"Train inputs:  {len(input_names)}")
print(f"Train targets: {len(target_names)}")
print(f"Test images:   {len(test_names)}")

assert len(input_names) == len(target_names), "Несоответствие файлов!"
assert set(input_names) == set(target_names), "Имена файлов не совпадают!"

# Визуализация примеров
print("\n📸 Примеры изображений:")
fig, axes = plt.subplots(3, 2, figsize=(12, 9))
for i in range(3):
    inp = load_image(os.path.join(TRAIN_INPUT_DIR, input_names[i]))
    tgt = load_image(os.path.join(TRAIN_TARGET_DIR, target_names[i]))

    axes[i, 0].imshow(inp)
    axes[i, 0].set_title(f"Вход: {input_names[i]}\n(перемешан + искажения)")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(tgt)
    axes[i, 1].set_title("Оригинал")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

print("""
💡 ВЫВОДЫ АНАЛИЗА:
1. Входные изображения — перемешанная сетка 24×24 фрагментов 20×20
2. Каждый фрагмент имеет ИНДИВИДУАЛЬНЫЕ искажения (яркость, контраст, шум, размытие, JPEG)
3. Это значит: даже при правильной сборке останутся ШВЫ между фрагментами

🎯 ВЫБРАННЫЙ ПОДХОД:
- Шаг 1: Сборка пазла (нейросеть предсказывает соседство фрагментов)
- Шаг 2: Реставрация (ансамбль из 3 архитектур убирает шум и швы)
- Шаг 3: Ансамбль для повышения стабильности предсказаний
""")

In [ ]:
# ==========================================================
# 4. БАЗОВЫЕ ФУНКЦИИ
# ==========================================================
def extract_fragments(img):
    """Разрезает изображение 480×480 на 576 фрагментов 20×20"""
    frags = []
    for r in range(GRID):
        for c in range(GRID):
            frags.append(img[r*FS:(r+1)*FS, c*FS:(c+1)*FS])
    return np.array(frags, dtype=np.uint8)

def make_pair_canvas(a, b, orientation="right"):
    """Создаёт канву 40×40 из двух фрагментов для нейросети"""
    canvas = np.zeros((40, 40, 3), dtype=np.uint8)
    if orientation == "right":
        canvas[0:20, 0:20] = a
        canvas[0:20, 20:40] = b
    else:
        canvas[0:20, 0:20] = a
        canvas[20:40, 0:20] = b
    return canvas

def grid_to_canvas(fragments, grid):
    """Собирает изображение 480×480 из сетки фрагментов"""
    canvas = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for r in range(GRID):
        for c in range(GRID):
            canvas[r*FS:(r+1)*FS, c*FS:(c+1)*FS] = fragments[grid[r, c]]
    return canvas

def calc_ssim(img1, img2):
    """Расчёт метрики SSIM (метрика конкурса)"""
    return ssim(img1, img2, channel_axis=2, data_range=255)

In [ ]:
# ==========================================================
# 5. ТОЧНЫЕ ИСКАЖЕНИЯ ПО УСЛОВИЮ КОНКУРСА
# ==========================================================
# Обоснование: модель должна учиться на тех же искажениях,
# что и в тесте. Иначе она не сможет обобщать.
#
# Из условия:
# - Яркость: ±30
# - Контраст: 0.70-1.30
# - Шум: σ = 40-55
# - Blur: Gaussian 3×3
# - JPEG quality: 35-50

def corrupt_fragment(frag):
    """Применяет к фрагменту искажения точно по условию"""
    x = frag.astype(np.float32)

    # 1. Контраст: 0.70-1.30
    contrast = np.random.uniform(0.70, 1.30)
    x = (x - 128.0) * contrast + 128.0

    # 2. Яркость: ±30
    x += np.random.uniform(-30.0, 30.0)
    x = np.clip(x, 0, 255).astype(np.uint8)

    # 3. Размытие: Gaussian 3×3
    if np.random.random() < 0.95:
        x = np.array(Image.fromarray(x).filter(ImageFilter.GaussianBlur(radius=1.0)))

    # 4. JPEG: quality 35-50
    if np.random.random() < 0.95:
        quality = np.random.randint(35, 51)
        img = Image.fromarray(x)
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=quality)
        buffer.seek(0)
        x = np.array(Image.open(buffer))

    # 5. Шум: σ = 40-55
    sigma = np.random.uniform(40.0, 55.0)
    noise = np.random.normal(0.0, sigma, x.shape).astype(np.float32)
    x = x.astype(np.float32) + noise
    x = np.clip(x, 0, 255).astype(np.uint8)

    return x

def make_dirty_assembled(clean_img):
    """Создаёт грязную собранную картинку из чистой"""
    frags = extract_fragments(clean_img)
    dirty = np.zeros_like(clean_img)
    idx = 0
    for r in range(GRID):
        for c in range(GRID):
            dirty[r*FS:(r+1)*FS, c*FS:(c+1)*FS] = corrupt_fragment(frags[idx])
            idx += 1
    return dirty

In [ ]:
# ==========================================================
# 6. РАЗБИЕНИЕ НА ОБУЧЕНИЕ И ВАЛИДАЦИЮ
# ==========================================================
all_names = input_names.copy()
random.shuffle(all_names)

val_names = all_names[:CONFIG["val_size"]]
train_names = all_names[CONFIG["val_size"]:]

print(f"Train: {len(train_names)} изображений")
print(f"Val:   {len(val_names)} изображений")

In [ ]:
# ==========================================================
# 7. МОДЕЛЬ ПАЗЛА (ОБОСНОВАНИЕ АРХИТЕКТУРЫ)
# ==========================================================
# Обоснование выбора архитектуры:
# - Задача: классификация пар фрагментов (3 класса)
# - Вход: канва 40×40×3 (два фрагмента рядом)
# - Выбрано: компактная CNN с BatchNorm для стабильности
# - Альтернативы: более глубокие сети не нужны, т.к. вход маленький

class PairCNN(nn.Module):
    """
    Модель для предсказания соседства фрагментов.
    0 = не соседи
    1 = второй справа от первого
    2 = второй снизу от первого
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        return self.head(x.view(x.size(0), -1))


class PuzzleDataset(Dataset):
    """
    Датасет пар фрагментов для обучения модели пазла.
    Ключевая особенность: фрагменты портятся на лету,
    чтобы модель училась работать с искажениями.
    """
    def __init__(self, names, target_dir, max_images=1000):
        self.frags_list = []
        self.items = []
        for name in names[:max_images]:
            img = load_image(os.path.join(target_dir, name))
            frags = extract_fragments(img)
            img_id = len(self.frags_list)
            self.frags_list.append(frags)

            # Позитивные пары (настоящие соседи)
            pos = []
            for r in range(GRID):
                for c in range(GRID):
                    idx = r * GRID + c
                    if c + 1 < GRID: pos.append((idx, idx+1, 1, "right"))
                    if r + 1 < GRID: pos.append((idx, idx+GRID, 2, "below"))

            # Негативные пары (случайные)
            neg = []
            for _ in range(len(pos)):
                i1, i2 = random.sample(range(GRID*GRID), 2)
                orient = random.choice(["right", "below"])
                neg.append((i1, i2, 0, orient))

            for i, j, lab, ori in pos: self.items.append((img_id, i, j, lab, ori))
            for i, j, lab, ori in neg: self.items.append((img_id, i, j, lab, ori))

        random.shuffle(self.items)
        print(f"Пар для обучения пазла: {len(self.items):,}")

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        img_id, i, j, label, orientation = self.items[idx]
        frags = self.frags_list[img_id]
        a = corrupt_fragment(frags[i])
        b = corrupt_fragment(frags[j])
        pair = make_pair_canvas(a, b, orientation)
        x = torch.from_numpy(pair.copy()).permute(2, 0, 1).float() / 255.0
        return x, int(label)


def train_puzzle_model(model, dataset, epochs, batch_size, name="model"):
    """Обучение модели пазла"""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        pbar = tqdm(loader, desc=f"{name} epoch {epoch+1}/{epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
            pbar.set_postfix(loss=total_loss/total, acc=correct/total)
    return model

In [ ]:
# ==========================================================
# 8. ОБУЧЕНИЕ МОДЕЛИ ПАЗЛА
# ==========================================================
print("=" * 50)
print("📐 ОБУЧЕНИЕ МОДЕЛИ ПАЗЛА")
print("=" * 50)

puzzle_dataset = PuzzleDataset(train_names, TRAIN_TARGET_DIR, CONFIG["puzzle_max_images"])
puzzle_model = PairCNN().to(device)

print(f"Параметров модели: {sum(p.numel() for p in puzzle_model.parameters()):,}")

puzzle_model = train_puzzle_model(
    puzzle_model, puzzle_dataset,
    CONFIG["puzzle_epochs"], CONFIG["puzzle_batch_size"], "Puzzle"
)

# Сохраняем веса (защита от потери при падении)
puzzle_weights = os.path.join(DATA_DIR, "puzzle_model.pth")
torch.save(puzzle_model.state_dict(), puzzle_weights)
print(f"✅ Веса сохранены: {puzzle_weights}")
torch.cuda.empty_cache()

In [ ]:
# ==========================================================
# 9. МОДЕЛИ РЕСТАВРАЦИИ (АНСАМБЛЬ ИЗ 3 АРХИТЕКТУР)
# ==========================================================
# Обоснование ансамбля:
# - Одна модель может переобучиться или давать артефакты
# - Усреднение предсказаний разных архитектур уменьшает шум
# - Выбраны 3 разные архитектуры для разнообразия:
#   1. UNet — хорошо восстанавливает детали
#   2. ResNet — стабильно убирает шум
#   3. WideUNet — быстрый и широкий

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
        )
    def forward(self, x): return self.block(x)

class RestorerUNet(nn.Module):
    """Классический UNet с residual-соединением"""
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(3, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.bot = ConvBlock(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = ConvBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ConvBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64)
        self.out = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bot(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return torch.clamp(x + self.out(d1), 0, 1)

class RestorerResNet(nn.Module):
    """ResNet с 8 блоками — стабильно убирает шум"""
    def __init__(self, num_blocks=8):
        super().__init__()
        self.conv_in = nn.Conv2d(3, 64, 3, padding=1)
        self.blocks = nn.Sequential(*[ConvBlock(64, 64) for _ in range(num_blocks)])
        self.conv_out = nn.Conv2d(64, 3, 3, padding=1)

    def forward(self, x):
        feat = torch.relu(self.conv_in(x))
        return torch.clamp(x + self.conv_out(self.blocks(feat)), 0, 1)

class RestorerWideUNet(nn.Module):
    """Широкий UNet — быстрый и эффективный"""
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(3, 64)
        self.enc2 = ConvBlock(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bot = ConvBlock(128, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ConvBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64)
        self.out = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bot(self.pool(e2))

        d2 = self.dec2(torch.cat([self.up2(b), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return torch.clamp(x + self.out(d1), 0, 1)


class RestorationDataset(Dataset):
    """Датасет для реставрации"""
    def __init__(self, names, target_dir, max_images=1000):
        self.names = names[:max_images]
        self.target_dir = target_dir
        print(f"Картинок для реставрации: {len(self.names)}")

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        clean = load_image(os.path.join(self.target_dir, self.names[idx]))
        dirty = make_dirty_assembled(clean)
        x = torch.from_numpy(dirty.copy()).permute(2, 0, 1).float() / 255.0
        y = torch.from_numpy(clean.copy()).permute(2, 0, 1).float() / 255.0
        return x, y


def train_restorer(model, dataset, epochs, batch_size, name="restorer"):
    """Обучение реставратора"""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.L1Loss()  # L1 лучше для SSIM, чем MSE

    for epoch in range(epochs):
        model.train()
        total_loss, total = 0, 0
        pbar = tqdm(loader, desc=f"{name} epoch {epoch+1}/{epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            total += x.size(0)
            pbar.set_postfix(loss=total_loss/total)
    return model

In [ ]:
# ==========================================================
# 10. ОБУЧЕНИЕ АНСАМБЛЯ РЕСТАВРАТОРОВ
# ==========================================================
print("=" * 50)
print("🎨 ОБУЧЕНИЕ АНСАМБЛЯ РЕСТАВРАТОРОВ")
print("=" * 50)

restore_dataset = RestorationDataset(train_names, TRAIN_TARGET_DIR, CONFIG["restore_max_images"])

restorers = []
restorer_classes = [
    (RestorerUNet, "UNet"),
    (RestorerResNet, "ResNet"),
    (RestorerWideUNet, "WideUNet"),
]

for cls, name in restorer_classes:
    print(f"\n📚 Обучение {name}...")
    model = cls().to(device)
    print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

    model = train_restorer(
        model, restore_dataset,
        CONFIG["restore_epochs"], CONFIG["restore_batch_size"], name
    )
    restorers.append(model)

    # Сохраняем веса
    weights_path = os.path.join(DATA_DIR, f"restorer_{name}.pth")
    torch.save(model.state_dict(), weights_path)
    print(f"✅ Веса сохранены: {weights_path}")

torch.cuda.empty_cache()

In [ ]:
# ==========================================================
# 11. АНСАМБЛЕВЫЙ ИНФЕРЕНС (СБОРКА + РЕСТАВРАЦИЯ)
# ==========================================================
@torch.no_grad()
def compute_scores(model, fragments, batch_size=2048):
    """Вычисляет матрицы вероятностей для всех пар фрагментов"""
    model.eval()
    n = len(fragments)
    right_scores = np.zeros((n, n), dtype=np.float32)
    below_scores = np.zeros((n, n), dtype=np.float32)

    for i in range(n):
        others = [j for j in range(n) if j != i]
        pairs_r = np.stack([make_pair_canvas(fragments[i], fragments[j], "right") for j in others])
        pairs_b = np.stack([make_pair_canvas(fragments[i], fragments[j], "below") for j in others])

        for pairs, scores, label in [(pairs_r, right_scores, 1), (pairs_b, below_scores, 2)]:
            x = torch.from_numpy(pairs).permute(0, 3, 1, 2).float() / 255.0
            probs = []
            for start in range(0, len(x), batch_size):
                out = model(x[start:start+batch_size].to(device))
                probs.append(torch.softmax(out, dim=1).cpu().numpy())
            probs = np.concatenate(probs)
            scores[i, others] = probs[:, label]

    return right_scores, below_scores


def choose_start_candidates(right_scores, below_scores, num_starts=30):
    """Выбирает кандидатов на верхний левый угол"""
    outgoing = right_scores.max(axis=1) + below_scores.max(axis=1)
    incoming = right_scores.max(axis=0) + below_scores.max(axis=0)
    corner_score = outgoing - incoming
    return np.argsort(-corner_score)[:num_starts]


def assemble_with_start(right_scores, below_scores, start_fragment):
    """Жадная сборка от конкретного стартового фрагмента"""
    n = right_scores.shape[0]
    grid = -np.ones((GRID, GRID), dtype=int)
    used = np.zeros(n, dtype=bool)
    grid[0, 0] = start_fragment
    used[start_fragment] = True

    for r in range(GRID):
        for c in range(GRID):
            if r == 0 and c == 0: continue
            scores = np.zeros(n, dtype=np.float32)
            count = 0
            if c > 0 and grid[r, c-1] >= 0:
                scores += right_scores[grid[r, c-1]]
                count += 1
            if r > 0 and grid[r-1, c] >= 0:
                scores += below_scores[grid[r-1, c]]
                count += 1
            if count > 0: scores /= count
            scores[used] = -1e18
            best_idx = int(np.argmax(scores))
            grid[r, c] = best_idx
            used[best_idx] = True

    # Оценка качества сборки
    total, cnt = 0.0, 0
    for r in range(GRID):
        for c in range(GRID):
            cur = grid[r, c]
            if c + 1 < GRID:
                total += right_scores[cur, grid[r, c+1]]
                cnt += 1
            if r + 1 < GRID:
                total += below_scores[cur, grid[r+1, c]]
                cnt += 1

    return grid, total / max(1, cnt)


def solve_puzzle(model, image, num_starts=30):
    """Сборка пазла с перебором кандидатов на угол"""
    fragments = extract_fragments(image)
    right_scores, below_scores = compute_scores(model, fragments)
    candidates = choose_start_candidates(right_scores, below_scores, num_starts)

    best_grid, best_score = None, -1e18
    for start_id in candidates:
        grid, score = assemble_with_start(right_scores, below_scores, int(start_id))
        if score > best_score:
            best_score, best_grid = score, grid

    return grid_to_canvas(fragments, best_grid)


@torch.no_grad()
def restore_ensemble(restorers, img):
    """Ансамбль реставраторов через усреднение"""
    x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    x = x.unsqueeze(0).to(device)
    outputs = []
    for model in restorers:
        model.eval()
        pred = model(x)
        outputs.append(pred.cpu().numpy()[0])
    avg = np.mean(outputs, axis=0).transpose(1, 2, 0)
    return np.clip(avg * 255, 0, 255).astype(np.uint8)

In [ ]:
# ==========================================================
# 12. ВАЛИДАЦИЯ И ЖУРНАЛ ЭКСПЕРИМЕНТОВ
# ==========================================================
EXPERIMENTS = []

def log_experiment(name, val_mean_ssim, notes=""):
    """Записывает эксперимент в журнал"""
    EXPERIMENTS.append({
        "time": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "name": name,
        "puzzle_images": CONFIG["puzzle_max_images"],
        "puzzle_epochs": CONFIG["puzzle_epochs"],
        "restore_images": CONFIG["restore_max_images"],
        "restore_epochs": CONFIG["restore_epochs"],
        "val_mean_ssim": round(val_mean_ssim, 4),
        "notes": notes,
    })
    df = pd.DataFrame(EXPERIMENTS).sort_values("val_mean_ssim", ascending=False)
    print("\n" + "=" * 60)
    print("📊 ЖУРНАЛ ЭКСПЕРИМЕНТОВ")
    print("=" * 60)
    print(df.to_string(index=False))
    df.to_csv(os.path.join(DATA_DIR, "experiments_table.csv"), index=False)

# Проверка на валидации
print("\n🔍 Проверка на валидации...")
val_scores = []
for name in val_names[:CONFIG["eval_images"]]:
    inp = load_image(os.path.join(TRAIN_INPUT_DIR, name))
    tgt = load_image(os.path.join(TRAIN_TARGET_DIR, name))

    assembled = solve_puzzle(puzzle_model, inp, CONFIG["puzzle_num_starts"])
    restored = restore_ensemble(restorers, assembled)

    score = calc_ssim(tgt, restored)
    val_scores.append(score)
    print(f"  {name}: SSIM = {score:.4f}")

mean_ssim = float(np.mean(val_scores))
print(f"\n✅ Mean Val SSIM: {mean_ssim:.4f}")

log_experiment("hybrid_final", mean_ssim, "1 puzzle + 3 restorers ensemble")

In [ ]:
# ==========================================================
# 13. УМНЫЙ САБМИТ (ЗАЩИЩЁН ОТ ПРЕРЫВАНИЙ)
# ==========================================================
def make_submission_smart(puzzle_model, restorers, test_names, out_dir, zip_path, full=True):
    """
    Создаёт сабмит, пропуская уже готовые файлы.
    Если процесс прервётся — при перезапуске продолжит с места остановки.
    """
    os.makedirs(out_dir, exist_ok=True)

    # Проверяем, что уже готово
    already_done = set(os.listdir(out_dir))

    names = test_names if full else test_names[:5]
    names_to_process = [n for n in names if n not in already_done]

    print(f"Уже готово: {len(already_done)}")
    print(f"Осталось сделать: {len(names_to_process)}")

    for name in tqdm(names_to_process):
        img = load_image(os.path.join(TEST_DIR, name))
        assembled = solve_puzzle(puzzle_model, img, CONFIG["submission_num_starts"])
        restored = restore_ensemble(restorers, assembled)
        Image.fromarray(restored).save(os.path.join(out_dir, name))

    # Пакуем ВСЁ в zip
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for filename in sorted(os.listdir(out_dir)):
            if filename.endswith(".png"):
                zf.write(os.path.join(out_dir, filename), arcname=filename)

    n_files = len([f for f in os.listdir(out_dir) if f.endswith(".png")])
    print(f"\n✅ Готово: {zip_path}")
    print(f"📦 Файлов в архиве: {n_files}")

    if full and n_files != len(test_names):
        print(f"⚠️ ВНИМАНИЕ: Ожидалось {len(test_names)}, получено {n_files}")
    elif full and n_files == len(test_names):
        print("🎉 Все файлы на месте!")

# Генерация сабмита
zip_path = os.path.join(DATA_DIR, "submission_final.zip")
make_submission_smart(
    puzzle_model, restorers, test_names,
    SUBMISSION_DIR, zip_path,
    full=RUN_FULL_SUBMISSION
)